# 🔬 LightGBM Overfit Lab — Credit Risk
## Fictitious Dataset: 700k rows · 20% Charge-Off Rate

This notebook walks through **detecting and fixing overfitting** in a LightGBM binary classifier step by step.

| Section | Topic |
|---------|-------|
| 0 | Generate fictitious credit-risk dataset |
| 1 | Baseline — intentional overfit + detection |
| 2 | Fix with Regularization |
| 3 | Fix with Subsampling |
| 4 | Fix with Early Stopping |
| 5 | Fix class imbalance (20% charge-off) |
| 6 | Stratified K-Fold Cross-Validation |
| 7 | Final comparison + best config |

## ⚙️ Install Dependencies

In [ ]:
# Run this cell once
!pip install lightgbm scikit-learn pandas numpy matplotlib seaborn imbalanced-learn -q

## 📦 Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (roc_auc_score, confusion_matrix,
                              precision_score, recall_score, f1_score)
import lightgbm as lgb

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
COLORS = {'train': '#2196F3', 'val': '#F44336', 'good': '#4CAF50', 'warn': '#FF9800', 'purple': '#9C27B0'}
print('✅ All imports successful')

---
## Section 0 — Generate Fictitious Credit Risk Dataset

We simulate a realistic credit risk dataset with:
- **24 features**: demographics, loan characteristics, credit history, behavioral signals
- **700,000 rows** with a logistic charge-off probability
- Target calibrated to exactly **20% charge-off rate**
- Split: **70% train / 15% val / 15% test** (stratified)

In [ ]:
N = 700_000
rng = np.random.default_rng(42)

# ── Demographics ────────────────────────────────────────────
age              = rng.integers(18, 75, N)
income           = np.clip(rng.lognormal(10.8, 0.6, N), 15_000, 500_000).astype(int)
employment_years = np.clip(rng.exponential(5, N), 0, 40)
num_dependents   = rng.integers(0, 6, N)
education_level  = rng.choice([0,1,2,3], N, p=[0.15,0.35,0.35,0.15])

# ── Loan Characteristics ────────────────────────────────────
loan_amount      = np.clip(rng.lognormal(10.0, 0.8, N), 1_000, 100_000).astype(int)
loan_term_months = rng.choice([12,24,36,48,60], N)
interest_rate    = np.clip(rng.normal(12, 4, N), 3, 30)
loan_purpose     = rng.choice([0,1,2,3,4], N)
debt_to_income   = np.clip(rng.beta(2, 5, N) * 0.8, 0.01, 0.75)

# ── Credit History ──────────────────────────────────────────
credit_score       = np.clip(rng.normal(680, 80, N), 300, 850).astype(int)
num_credit_lines   = rng.integers(1, 20, N)
oldest_account_yrs = np.clip(rng.exponential(8, N), 0, 40)
num_late_payments  = rng.integers(0, 15, N)
num_hard_inquiries = rng.integers(0, 10, N)
credit_utilization = np.clip(rng.beta(2, 4, N), 0.01, 0.99)
has_bankruptcy     = rng.choice([0,1], N, p=[0.93, 0.07])
has_collections    = rng.choice([0,1], N, p=[0.88, 0.12])

# ── Behavioral / Derived ────────────────────────────────────
months_employed      = employment_years * 12
payment_history_pct  = np.clip(1 - (num_late_payments / (num_credit_lines + 1)) * 0.3, 0.3, 1.0)
revolving_balance    = (credit_utilization * income * 0.2).astype(int)
savings_balance      = np.clip(rng.lognormal(8, 1.5, N), 0, 200_000).astype(int)
checking_balance     = np.clip(rng.lognormal(7, 1.2, N), 0, 50_000).astype(int)
monthly_payment_est  = loan_amount / loan_term_months
payment_to_income    = monthly_payment_est / (income / 12)

# ── Target: charge-off ──────────────────────────────────────
log_odds = (
    -4.5
    + (700 - credit_score) * 0.008
    + debt_to_income * 3.5
    + num_late_payments * 0.15
    + has_bankruptcy * 1.8
    + has_collections * 0.9
    + credit_utilization * 1.2
    + payment_to_income * 2.0
    - payment_history_pct * 1.5
    - (income / 100_000) * 0.8
    + num_hard_inquiries * 0.1
    - oldest_account_yrs * 0.02
    + rng.normal(0, 0.5, N)
)
prob_chargeoff = 1 / (1 + np.exp(-log_odds))
threshold_co   = np.percentile(prob_chargeoff, 80)
target         = (prob_chargeoff > threshold_co).astype(int)

# ── Assemble DataFrame ──────────────────────────────────────
df = pd.DataFrame({
    'age': age, 'income': income, 'employment_years': employment_years,
    'num_dependents': num_dependents, 'education_level': education_level,
    'loan_amount': loan_amount, 'loan_term_months': loan_term_months,
    'interest_rate': interest_rate, 'loan_purpose': loan_purpose,
    'debt_to_income': debt_to_income, 'credit_score': credit_score,
    'num_credit_lines': num_credit_lines, 'oldest_account_yrs': oldest_account_yrs,
    'num_late_payments': num_late_payments, 'num_hard_inquiries': num_hard_inquiries,
    'credit_utilization': credit_utilization, 'has_bankruptcy': has_bankruptcy,
    'has_collections': has_collections, 'months_employed': months_employed,
    'payment_history_pct': payment_history_pct, 'revolving_balance': revolving_balance,
    'savings_balance': savings_balance, 'checking_balance': checking_balance,
    'payment_to_income': payment_to_income,
    'charge_off': target
})

print(f'Dataset shape : {df.shape}')
print(f'Charge-off rate: {df.charge_off.mean():.1%}')
df.head(3)

In [ ]:
FEATURES = [c for c in df.columns if c != 'charge_off']
X, y = df[FEATURES], df['charge_off']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)
X_val,   X_test, y_val,   y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

train_data = lgb.Dataset(X_train, label=y_train)
val_data   = lgb.Dataset(X_val,   label=y_val, reference=train_data)

print(f'Train : {len(X_train):>7,} rows  charge-off: {y_train.mean():.1%}')
print(f'Val   : {len(X_val):>7,} rows  charge-off: {y_val.mean():.1%}')
print(f'Test  : {len(X_test):>7,} rows  charge-off: {y_test.mean():.1%}')

In [ ]:
# Quick EDA — charge-off rate by credit score bucket
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(df[df.charge_off==0]['credit_score'], bins=40, alpha=0.6, color=COLORS['good'], label='No charge-off')
axes[0].hist(df[df.charge_off==1]['credit_score'], bins=40, alpha=0.6, color=COLORS['val'],  label='Charge-off')
axes[0].set_title('Credit Score Distribution'); axes[0].legend()

axes[1].hist(df[df.charge_off==0]['debt_to_income'], bins=40, alpha=0.6, color=COLORS['good'])
axes[1].hist(df[df.charge_off==1]['debt_to_income'], bins=40, alpha=0.6, color=COLORS['val'])
axes[1].set_title('Debt-to-Income Distribution')

axes[2].bar(['No Charge-Off (80%)', 'Charge-Off (20%)'],
            [df.charge_off.value_counts()[0], df.charge_off.value_counts()[1]],
            color=[COLORS['good'], COLORS['val']], alpha=0.8)
axes[2].set_title('Class Distribution')

plt.suptitle('Section 0 — Dataset EDA', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## Section 1 — Baseline: Intentional Overfit + Detection

### 🔴 Problem
A common mistake is running LightGBM with high complexity and no constraints:
- `num_leaves=400` → huge trees with many splits
- `min_child_samples=5` → leaves can contain just 5 samples (memorisation)
- `max_depth=-1` → no depth limit at all

### 🔍 How to detect overfit
1. **Train/Val AUC gap** — if Train AUC >> Val AUC, the model is memorising
2. **Training curves** — val AUC peaks early then degrades while train keeps rising
3. **Gap threshold**: gap > 0.05 = severe | 0.02–0.05 = moderate | < 0.02 = healthy

In [ ]:
# Helper: evaluate a model on all 3 splits
def evaluate(model, tag=''):
    train_auc = roc_auc_score(y_train, model.predict(X_train))
    val_auc   = roc_auc_score(y_val,   model.predict(X_val))
    test_auc  = roc_auc_score(y_test,  model.predict(X_test))
    gap       = train_auc - val_auc
    flag = '⚠️  SEVERE' if gap > 0.05 else ('🟡 MODERATE' if gap > 0.02 else '✅ HEALTHY')
    print(f'  {tag:<28} Train={train_auc:.4f}  Val={val_auc:.4f}  Test={test_auc:.4f}  Gap={gap:.4f}  {flag}')
    return {'train': train_auc, 'val': val_auc, 'test': test_auc, 'gap': gap}

print('evaluate() helper ready')

In [ ]:
params_baseline = {
    'objective': 'binary', 'metric': 'auc', 'verbosity': -1,
    'num_leaves': 400,       # ← TOO HIGH — complex trees
    'max_depth': -1,         # ← NO LIMIT — trees grow unconstrained
    'min_child_samples': 5,  # ← TOO LOW — memorises small patterns
    'learning_rate': 0.1,
}

evals_baseline = {}
model_baseline = lgb.train(
    params_baseline, train_data,
    num_boost_round=500,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'val'],
    callbacks=[lgb.record_evaluation(evals_baseline), lgb.log_evaluation(100)],
)
res_baseline = evaluate(model_baseline, 'Baseline (overfit)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training curves — the key overfit detection visual
ax = axes[0]
ax.plot(evals_baseline['train']['auc'], color=COLORS['train'], lw=2, label='Train AUC')
ax.plot(evals_baseline['val']['auc'],   color=COLORS['val'],   lw=2, label='Val AUC')
ax.fill_between(range(500),
                evals_baseline['train']['auc'],
                evals_baseline['val']['auc'],
                alpha=0.15, color=COLORS['val'], label='Overfit Gap')
ax.set_title('🔴 Overfit Detected: Train >> Val AUC', fontsize=12)
ax.set_xlabel('Iteration'); ax.set_ylabel('AUC'); ax.legend(); ax.grid(alpha=0.3)

# Feature importance
ax2 = axes[1]
fi = pd.Series(model_baseline.feature_importance(importance_type='gain'), index=FEATURES).nlargest(15)
fi.sort_values().plot(kind='barh', ax=ax2, color=COLORS['train'], alpha=0.8)
ax2.set_title('Top 15 Feature Importances (Gain)', fontsize=12)
ax2.grid(alpha=0.3, axis='x')

plt.suptitle(f"Section 1 — Baseline | Train={res_baseline['train']:.4f}  Val={res_baseline['val']:.4f}  Gap={res_baseline['gap']:.4f}", fontsize=11)
plt.tight_layout()
plt.show()
print(f"\n⚠️  Overfit gap = {res_baseline['gap']:.4f} — model is memorising training data!")

---
## Section 2 — Fix #1: Regularization

Constrain tree complexity directly so each tree generalises better:

| Parameter | Baseline | Fixed | Effect |
|-----------|----------|-------|--------|
| `num_leaves` | 400 | **31** | Fewer leaf nodes per tree |
| `max_depth` | -1 | **6** | Hard depth ceiling |
| `min_child_samples` | 5 | **100** | Each leaf needs ≥100 samples |
| `lambda_l1` | 0 | **0.1** | L1 penalty on leaf weights |
| `lambda_l2` | 0 | **0.1** | L2 penalty on leaf weights |
| `min_gain_to_split` | 0 | **0.01** | Skip splits with tiny gain |

In [ ]:
params_reg = {
    'objective': 'binary', 'metric': 'auc', 'verbosity': -1,
    'num_leaves': 31,            # ← down from 400
    'max_depth': 6,              # ← explicit cap
    'min_child_samples': 100,    # ← up from 5
    'lambda_l1': 0.1,
    'lambda_l2': 0.1,
    'min_gain_to_split': 0.01,
    'learning_rate': 0.05,
}

evals_reg = {}
model_reg = lgb.train(
    params_reg, train_data,
    num_boost_round=500,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'val'],
    callbacks=[lgb.record_evaluation(evals_reg), lgb.log_evaluation(100)],
)
res_reg = evaluate(model_reg, 'Regularized')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, evals, res, title in [
    (axes[0], evals_baseline, res_baseline, f"Before — Gap={res_baseline['gap']:.4f} ⚠️"),
    (axes[1], evals_reg,      res_reg,      f"After  — Gap={res_reg['gap']:.4f} ✅"),
]:
    ax.plot(evals['train']['auc'], color=COLORS['train'], lw=2, label='Train')
    ax.plot(evals['val']['auc'],   color=COLORS['val'],   lw=2, label='Val')
    ax.fill_between(range(len(evals['train']['auc'])),
                    evals['train']['auc'], evals['val']['auc'], alpha=0.1, color=COLORS['val'])
    ax.set_title(title, fontsize=12); ax.set_xlabel('Iteration'); ax.set_ylabel('AUC')
    ax.legend(); ax.grid(alpha=0.3); ax.set_ylim(0.65, 1.0)

reduction = (1 - res_reg['gap'] / res_baseline['gap']) * 100
plt.suptitle(f'Section 2 — Regularization: Gap reduced by {reduction:.0f}%', fontsize=12)
plt.tight_layout()
plt.show()

---
## Section 3 — Fix #2: Subsampling (Stochastic Boosting)

Add controlled randomness per tree to reduce variance:
- **`subsample=0.8`** — each tree trains on a random 80% of rows
- **`colsample_bytree=0.8`** — each tree uses a random 80% of features

> With 490k training rows, even `subsample=0.5` = 245k samples per tree — more than enough.

We sweep `subsample` from 0.3 to 1.0 to find the sweet spot.

In [ ]:
params_sub_base = {
    'objective': 'binary', 'metric': 'auc', 'verbosity': -1,
    'num_leaves': 63, 'max_depth': 7, 'min_child_samples': 50,
    'colsample_bytree': 0.8, 'learning_rate': 0.05,
}

sub_results = {}
for ss in [0.3, 0.5, 0.7, 0.9, 1.0]:
    p = {**params_sub_base, 'subsample': ss, 'subsample_freq': 1 if ss < 1.0 else 0}
    ev = {}
    m  = lgb.train(p, train_data, num_boost_round=300,
                   valid_sets=[train_data, val_data], valid_names=['train','val'],
                   callbacks=[lgb.record_evaluation(ev), lgb.log_evaluation(-1)])
    gap = roc_auc_score(y_train, m.predict(X_train)) - roc_auc_score(y_val, m.predict(X_val))
    sub_results[ss] = {'gap': gap, 'val_auc': roc_auc_score(y_val, m.predict(X_val)),
                       'train_curves': ev['train']['auc'], 'val_curves': ev['val']['auc']}
    print(f'  subsample={ss:.1f}  val_auc={sub_results[ss]["val_auc"]:.4f}  gap={gap:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Gap vs subsample rate
ax = axes[0]
ss_vals  = list(sub_results.keys())
gap_vals = [sub_results[s]['gap']     for s in ss_vals]
val_vals = [sub_results[s]['val_auc'] for s in ss_vals]
ax.plot(ss_vals, gap_vals, 'o-', color=COLORS['val'],   lw=2, label='Overfit Gap')
ax.plot(ss_vals, val_vals, 's-', color=COLORS['train'], lw=2, label='Val AUC')
ax.set_title('subsample Rate vs. Performance'); ax.set_xlabel('subsample')
ax.legend(); ax.grid(alpha=0.3)

# Training curves for best subsample (0.8)
best_ss = min(sub_results, key=lambda s: sub_results[s]['gap'])
ax = axes[1]
ax.plot(sub_results[best_ss]['train_curves'], color=COLORS['train'], lw=2, label=f'Train (ss={best_ss})')
ax.plot(sub_results[best_ss]['val_curves'],   color=COLORS['val'],   lw=2, label=f'Val   (ss={best_ss})')
ax.set_title(f'Best Subsample = {best_ss}'); ax.set_xlabel('Iteration'); ax.set_ylabel('AUC')
ax.legend(); ax.grid(alpha=0.3)

# Comparison bar
ax = axes[2]
ax.bar([str(s) for s in ss_vals], gap_vals,
       color=[COLORS['val'] if g > 0.05 else (COLORS['warn'] if g > 0.02 else COLORS['good']) for g in gap_vals],
       alpha=0.8, edgecolor='white')
ax.axhline(0.05, color=COLORS['val'],  lw=1.5, linestyle='--', label='Severe threshold')
ax.axhline(0.02, color=COLORS['good'], lw=1.5, linestyle='--', label='Healthy threshold')
ax.set_title('Overfit Gap per Subsample Rate'); ax.set_xlabel('subsample'); ax.set_ylabel('Gap')
ax.legend(); ax.grid(alpha=0.3, axis='y')

plt.suptitle('Section 3 — Subsampling', fontsize=12)
plt.tight_layout()
plt.show()

---
## Section 4 — Fix #3: Early Stopping

Set `n_estimators` high (ceiling of 2000) and let early stopping decide when to stop.

```python
lgb.early_stopping(stopping_rounds=50)  # stop if val AUC doesn't improve for 50 rounds
```

**Benefits:**
- No need to manually tune `n_estimators`
- Prevents fitting noise in later iterations
- Works best with a lower learning rate (0.05)

In [ ]:
params_es = {
    'objective': 'binary', 'metric': 'auc', 'verbosity': -1,
    'num_leaves': 63, 'max_depth': 6, 'min_child_samples': 50,
    'subsample': 0.8, 'subsample_freq': 1, 'colsample_bytree': 0.8,
    'lambda_l1': 0.05, 'lambda_l2': 0.05,
    'learning_rate': 0.05,
}

evals_es = {}
model_es = lgb.train(
    params_es,
    train_data,
    num_boost_round=2000,          # ← high ceiling
    valid_sets=[train_data, val_data],
    valid_names=['train', 'val'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=True),  # ← key callback
        lgb.record_evaluation(evals_es),
        lgb.log_evaluation(100),
    ],
)

best_iter  = model_es.best_iteration
final_iter = len(evals_es['val']['auc'])
print(f'\nBest iteration : {best_iter}')
print(f'Stopped at     : {final_iter}  (saved {2000 - final_iter} unused trees)')
res_es = evaluate(model_es, 'Early Stopping')

In [ ]:
# Effect of different stopping_rounds values
print('stopping_rounds comparison:')
print(f'{"rounds":<8} {"best_iter":<12} {"val_auc"}')
for sr in [10, 25, 50, 100, 200]:
    ev2 = {}
    m2 = lgb.train({**params_es}, train_data, num_boost_round=2000,
                   valid_sets=[train_data, val_data], valid_names=['train','val'],
                   callbacks=[lgb.early_stopping(sr, verbose=False),
                               lgb.record_evaluation(ev2), lgb.log_evaluation(-1)])
    v = max(ev2['val']['auc'])
    print(f'{sr:<8} {m2.best_iteration:<12} {v:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# With early stopping
ax = axes[0]
iters = range(len(evals_es['train']['auc']))
ax.plot(iters, evals_es['train']['auc'], color=COLORS['train'], lw=2, label='Train AUC')
ax.plot(iters, evals_es['val']['auc'],   color=COLORS['val'],   lw=2, label='Val AUC')
ax.axvline(best_iter,    color=COLORS['good'], lw=2, linestyle='--', label=f'Best iter ({best_iter})')
ax.axvline(final_iter-1, color=COLORS['warn'], lw=2, linestyle=':',  label=f'Stopped ({final_iter})')
ax.set_title('With Early Stopping', fontsize=12)
ax.set_xlabel('Iteration'); ax.set_ylabel('AUC'); ax.legend(); ax.grid(alpha=0.3)

# Without early stopping (baseline — trained 500)
ax = axes[1]
iters_b = range(len(evals_baseline['train']['auc']))
ax.plot(iters_b, evals_baseline['train']['auc'], color=COLORS['train'], lw=2, label='Train')
ax.plot(iters_b, evals_baseline['val']['auc'],   color=COLORS['val'],   lw=2, label='Val')
ax.fill_between(iters_b, evals_baseline['train']['auc'], evals_baseline['val']['auc'],
                alpha=0.15, color=COLORS['val'], label='Overfit region')
ax.set_title('Without Early Stopping (500 iters, overfit)', fontsize=12)
ax.set_xlabel('Iteration'); ax.set_ylabel('AUC'); ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Section 4 — Early Stopping', fontsize=12)
plt.tight_layout()
plt.show()

---
## Section 5 — Fix #4: Class Imbalance (20% charge-off)

A naive model maximises accuracy by almost always predicting 'no charge-off'.
For credit risk we need high **recall** on the minority class.

| Strategy | How it works |
|----------|--------------|
| No correction | Baseline — biased toward majority |
| `scale_pos_weight=4` | Weights minority class 4× (≈ 80/20 ratio) |
| `is_unbalance=True` | LightGBM auto-computes weights |
| SMOTE | Generates synthetic minority samples |

In [ ]:
from imblearn.over_sampling import SMOTE

base_params = {
    'objective': 'binary', 'metric': 'auc', 'verbosity': -1,
    'num_leaves': 31, 'max_depth': 6, 'min_child_samples': 100,
    'subsample': 0.8, 'subsample_freq': 1, 'colsample_bytree': 0.8,
    'lambda_l1': 0.1, 'lambda_l2': 0.1, 'learning_rate': 0.05,
}

imb_results = {}
models_preds = []

# 1. No correction
m1 = lgb.train({**base_params}, train_data, num_boost_round=300, callbacks=[lgb.log_evaluation(-1)])
p1 = (m1.predict(X_val) > 0.5).astype(int)
imb_results['No correction'] = {
    'auc': roc_auc_score(y_val, m1.predict(X_val)), 'prec': precision_score(y_val, p1, zero_division=0),
    'rec': recall_score(y_val, p1, zero_division=0), 'f1': f1_score(y_val, p1, zero_division=0)}
models_preds.append(('No correction', p1))

# 2. scale_pos_weight
m2 = lgb.train({**base_params, 'scale_pos_weight': 4}, train_data, num_boost_round=300, callbacks=[lgb.log_evaluation(-1)])
p2 = (m2.predict(X_val) > 0.5).astype(int)
imb_results['scale_pos_weight=4'] = {
    'auc': roc_auc_score(y_val, m2.predict(X_val)), 'prec': precision_score(y_val, p2, zero_division=0),
    'rec': recall_score(y_val, p2, zero_division=0), 'f1': f1_score(y_val, p2, zero_division=0)}
models_preds.append(('scale_pos_weight=4', p2))

# 3. is_unbalance
m3 = lgb.train({**base_params, 'is_unbalance': True}, train_data, num_boost_round=300, callbacks=[lgb.log_evaluation(-1)])
p3 = (m3.predict(X_val) > 0.5).astype(int)
imb_results['is_unbalance=True'] = {
    'auc': roc_auc_score(y_val, m3.predict(X_val)), 'prec': precision_score(y_val, p3, zero_division=0),
    'rec': recall_score(y_val, p3, zero_division=0), 'f1': f1_score(y_val, p3, zero_division=0)}
models_preds.append(('is_unbalance=True', p3))

# 4. SMOTE (on 50k subsample for speed)
X_tr_sm, y_tr_sm = X_train.iloc[:50_000], y_train.iloc[:50_000]
X_res, y_res = SMOTE(random_state=42, k_neighbors=5).fit_resample(X_tr_sm, y_tr_sm)
td_sm = lgb.Dataset(X_res, label=y_res)
m4 = lgb.train({**base_params}, td_sm, num_boost_round=300, callbacks=[lgb.log_evaluation(-1)])
p4 = (m4.predict(X_val) > 0.5).astype(int)
imb_results['SMOTE'] = {
    'auc': roc_auc_score(y_val, m4.predict(X_val)), 'prec': precision_score(y_val, p4, zero_division=0),
    'rec': recall_score(y_val, p4, zero_division=0), 'f1': f1_score(y_val, p4, zero_division=0)}
models_preds.append(('SMOTE', p4))

# Print comparison table
print(f'\n  {"Strategy":<22} {"AUC":>6} {"Precision":>10} {"Recall":>8} {"F1":>6}')
print('  ' + '-'*55)
for name, r in imb_results.items():
    print(f'  {name:<22} {r["auc"]:>6.4f} {r["prec"]:>10.4f} {r["rec"]:>8.4f} {r["f1"]:>6.4f}')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, (name, preds) in zip(axes, models_preds):
    cm = confusion_matrix(y_val, preds, normalize='true')
    sns.heatmap(cm, annot=True, fmt='.2%', ax=ax, cmap='Blues',
                xticklabels=['Pred 0', 'Pred 1'],
                yticklabels=['True 0', 'True 1'],
                linewidths=0.5, cbar=False)
    r = imb_results[name]
    ax.set_title(f'{name}\nF1={r["f1"]:.3f}  Recall={r["rec"]:.3f}', fontsize=10)

plt.suptitle('Section 5 — Confusion Matrices: Imbalance Strategies', fontsize=12)
plt.tight_layout()
plt.show()

---
## Section 6 — Stratified K-Fold Cross-Validation

A single train/val split can be lucky or unlucky. **5-fold stratified CV** gives a more reliable generalisation estimate.

Each fold preserves the 20% charge-off ratio via `stratify=True`.

In [ ]:
best_params = {
    'objective': 'binary', 'metric': 'auc', 'verbosity': -1,
    'num_leaves': 31, 'max_depth': 6, 'min_child_samples': 100,
    'subsample': 0.8, 'subsample_freq': 1, 'colsample_bytree': 0.8,
    'lambda_l1': 0.1, 'lambda_l2': 0.1,
    'scale_pos_weight': 4,
    'learning_rate': 0.05,
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_aucs, fold_gaps = [], []

print(f'{"Fold":<6} {"Train AUC":>10} {"Val AUC":>9} {"Gap":>8}')
print('-' * 36)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    Xtr, Xva = X.iloc[tr_idx], X.iloc[va_idx]
    ytr, yva = y.iloc[tr_idx], y.iloc[va_idx]
    td = lgb.Dataset(Xtr, label=ytr)
    vd = lgb.Dataset(Xva, label=yva, reference=td)
    ev = {}
    m = lgb.train(best_params, td, num_boost_round=500,
                  valid_sets=[td, vd], valid_names=['train','val'],
                  callbacks=[lgb.early_stopping(50, verbose=False),
                             lgb.record_evaluation(ev), lgb.log_evaluation(-1)])
    tr_auc = roc_auc_score(ytr, m.predict(Xtr))
    va_auc = roc_auc_score(yva, m.predict(Xva))
    fold_aucs.append(va_auc); fold_gaps.append(tr_auc - va_auc)
    print(f'Fold {fold+1}    {tr_auc:.4f}     {va_auc:.4f}   {tr_auc - va_auc:.4f}')

print(f'\nMean Val AUC : {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}')
print(f'Mean Gap     : {np.mean(fold_gaps):.4f} ± {np.std(fold_gaps):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.bar(range(1, 6), fold_aucs, color=COLORS['train'], alpha=0.8, edgecolor='white')
ax.axhline(np.mean(fold_aucs), color=COLORS['warn'], lw=2, linestyle='--',
           label=f'Mean = {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}')
ax.set_xticks(range(1, 6)); ax.set_xticklabels([f'Fold {i}' for i in range(1, 6)])
ax.set_title('Val AUC per Fold'); ax.set_ylabel('AUC')
ax.legend(); ax.grid(alpha=0.3, axis='y'); ax.set_ylim(0.80, 0.92)

ax = axes[1]
ax.bar(range(1, 6), fold_gaps, color=COLORS['val'], alpha=0.8, edgecolor='white')
ax.axhline(np.mean(fold_gaps), color=COLORS['warn'], lw=2, linestyle='--',
           label=f'Mean gap = {np.mean(fold_gaps):.4f}')
ax.axhline(0.02, color=COLORS['good'], lw=1.5, linestyle=':', label='Healthy threshold')
ax.set_xticks(range(1, 6)); ax.set_xticklabels([f'Fold {i}' for i in range(1, 6)])
ax.set_title('Overfit Gap per Fold'); ax.set_ylabel('Train AUC − Val AUC')
ax.legend(); ax.grid(alpha=0.3, axis='y')

plt.suptitle('Section 6 — 5-Fold Stratified CV', fontsize=12)
plt.tight_layout()
plt.show()

---
## Section 7 — Final Comparison + Best Config

All experiments side by side — val AUC and overfit gap.

In [ ]:
res_reg  = evaluate(model_reg,  'Regularization')
res_sub_best = {'train': None, 'val': sub_results[min(sub_results, key=lambda s: sub_results[s]['gap'])]['val_auc'],
                'test': None,  'gap': sub_results[min(sub_results, key=lambda s: sub_results[s]['gap'])]['gap']}
res_es   = evaluate(model_es,   'Early Stopping')

all_results = {
    'Baseline (overfit)' : res_baseline,
    'Regularization'     : res_reg,
    'Subsampling (best)' : res_sub_best,
    'Early Stopping'     : res_es,
    'CV Mean'            : {'train': None, 'val': np.mean(fold_aucs), 'test': None, 'gap': np.mean(fold_gaps)},
}

print(f'\n{"Model":<24} {"Val AUC":>9} {"Overfit Gap":>13}')
print('-' * 50)
for name, r in all_results.items():
    flag = '⚠️ ' if r['gap'] > 0.05 else ('✅' if r['gap'] < 0.02 else '🟡')
    print(f'{name:<24} {r["val"]:>9.4f} {r["gap"]:>10.4f}  {flag}')

In [ ]:
names     = list(all_results.keys())
val_aucs  = [r['val'] for r in all_results.values()]
gaps      = [r['gap'] for r in all_results.values()]
bar_cols  = [COLORS['val'] if g > 0.05 else (COLORS['warn'] if g > 0.02 else COLORS['good']) for g in gaps]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].barh(names, val_aucs, color=bar_cols, alpha=0.85, edgecolor='white')
for i, v in enumerate(val_aucs):
    axes[0].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=10)
axes[0].set_title('Val AUC by Experiment'); axes[0].set_xlabel('AUC')
axes[0].grid(alpha=0.3, axis='x'); axes[0].set_xlim(0.78, 0.93)

axes[1].barh(names, gaps, color=bar_cols, alpha=0.85, edgecolor='white')
for i, v in enumerate(gaps):
    axes[1].text(v + 0.0005, i, f'{v:.4f}', va='center', fontsize=10)
axes[1].axvline(0.05, color=COLORS['val'],  lw=1.5, linestyle='--', label='Severe threshold')
axes[1].axvline(0.02, color=COLORS['good'], lw=1.5, linestyle='--', label='Healthy threshold')
axes[1].set_title('Overfit Gap (lower = better)'); axes[1].set_xlabel('Train AUC − Val AUC')
axes[1].legend(); axes[1].grid(alpha=0.3, axis='x')

plt.suptitle('Section 7 — All Experiments Final Comparison', fontsize=12)
plt.tight_layout()
plt.show()

---
## ✅ Recommended Final Config (Copy-Paste Ready)

Combines all fixes: regularization + subsampling + early stopping + imbalance handling.


In [ ]:
FINAL_PARAMS = {
    'objective': 'binary',
    'metric': 'auc',
    'verbosity': -1,
    # Tree complexity
    'num_leaves': 31,
    'max_depth': 6,
    'min_child_samples': 100,
    # Regularization
    'lambda_l1': 0.1,
    'lambda_l2': 0.1,
    'min_gain_to_split': 0.01,
    # Stochastic boosting
    'subsample': 0.8,
    'subsample_freq': 1,
    'colsample_bytree': 0.8,
    # Class imbalance (20% charge-off)
    'scale_pos_weight': 4,
    # Learning
    'learning_rate': 0.05,
    'random_state': 42,
}

evals_final = {}
model_final = lgb.train(
    FINAL_PARAMS,
    train_data,
    num_boost_round=2000,
    valid_sets=[val_data],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.record_evaluation(evals_final),
        lgb.log_evaluation(100),
    ],
)

evaluate(model_final, 'FINAL MODEL')